In [1]:
import os
from functools import partial

import xgboost as xgb
from imblearn.over_sampling import SMOTE, SVMSMOTE, BorderlineSMOTE, KMeansSMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from src.training_utils import (
    RANDOM_SEED,
    BinaryDatasetManager,
    ParamRunner,
    MyBaggingClassifier,
)
from src.utils import RESULTS_DIR

In [2]:
data_manager = BinaryDatasetManager()

data_comphrasion_df = data_manager.df[
    [
        "ratio (0 : 1)",
        "shape",
        "inbalance strength",
        "class negative num",
        "class positive num",
    ]
]
data_comphrasion_df

,ratio (0 : 1),shape,inbalance strength,class negative num,class positive num
ecoli,43:5,"(336, 7)",8.60,301,35
optical_digits,2533:277,"(5620, 64)",9.14,5066,554
satimage,5809:626,"(6435, 36)",9.28,5809,626
pen_digits,9937:1055,"(10992, 16)",9.42,9937,1055
abalone,3786:391,"(4177, 10)",9.68,3786,391
sick_euthyroid,2870:293,"(3163, 42)",9.80,2870,293
spectrometer,54:5,"(531, 93)",10.80,486,45
car_eval_34,797:67,"(1728, 21)",11.90,1594,134
isolet,2399:200,"(7797, 617)",12.00,7197,600
us_crime,922:75,"(1994, 100)",12.29,1844,150


In [3]:
data_comphrasion_df.to_latex(
    os.path.join(RESULTS_DIR, "data_comphrasion_df.tex"),
    float_format="%.2f",
    column_format="lcccc",
    escape=False,
)

# Setup

In [4]:
smote_kwargs = {
    "sampling_strategy": "minority",
    "k_neighbors": 5,
}
m_neighbors = 10

smote_class = partial(
    SMOTE,
    **smote_kwargs,
)

borderline_smote_class = partial(
    BorderlineSMOTE, **smote_kwargs, kind="borderline-1", m_neighbors=m_neighbors
)

kmeans_smote_class = partial(KMeansSMOTE, **smote_kwargs, m_neighbors=m_neighbors)

svm_smote_class = partial(SVMSMOTE, **smote_kwargs, m_neighbors=m_neighbors)

base_bagging_classifier_class = partial(
    MyBaggingClassifier,
    bootstrap=True,
)

SMOTE_CLASSES = {
    "SMOTE": smote_class,
    "BorderlineSMOTE": borderline_smote_class,
    "KMeansSMOTE": kmeans_smote_class,
    "SVMSMOTE": svm_smote_class,
}

# Trees

In [5]:
tree_kwargs = {
    "criterion": "gini",
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": None,  # all features
}

tree_class = partial(
    DecisionTreeClassifier,
    **tree_kwargs,
)

random_forest_class = partial(
    RandomForestClassifier,
    n_jobs=-1,
    **tree_kwargs,
)

xgb_class = partial(
    xgb.XGBClassifier,
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    device="gpu",
)

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "roc_auc": "roc_auc",
}

bagging_param_grid = {
    "n_estimators": [25, 50, 100, 250, 500],
    "max_depth": [3, 5, 10],
}

In [6]:
run_id = 0
n_takes = 5
n_jobs = -1

results_dir = os.path.join(RESULTS_DIR, f"run_{run_id}")
os.makedirs(results_dir, exist_ok=True)
performed_runs = set(os.listdir(results_dir))
for run in performed_runs:
    if run.startswith("run_"):
        run_id = max(run_id, int(run.split("_")[1]))

In [7]:
def finish(runner, X_train, y_train, X_test, y_test, smote_class):
    """Finish the test run."""
    runner.fit(X_train, y_train, X_test, y_test)

    runner.results_.to_json(
        os.path.join(results_dir, filename),
        orient="records",
        lines=True,
    )

# Smote + Bagging same for each model

In [8]:
model_name = "DecisionTreeClassifier"

for smote_name, smote_class in sorted(SMOTE_CLASSES.items(), key=lambda x: x[0]):
    for dataset_name, (
        (X_train, y_train),
        (_, _),
        (X_test, y_test),
    ) in data_manager.serve(oversampler=smote_class()):
        for take in range(1, n_takes + 1):
            random_state = RANDOM_SEED + take
            filename = f"{dataset_name}__{smote_name}__{model_name}__bagging_simple__{take}.json"
            if filename in performed_runs:
                print(
                    f"Already performed {smote_name} with {model_name} on {dataset_name} take {take}"
                )
                continue
            print(
                f"Performing {smote_name} with {model_name} on {dataset_name} take {take}"
            )

            runner = ParamRunner(
                base_estimator_class=tree_class,
                bagging_classifier_class=base_bagging_classifier_class,
                param_grid=bagging_param_grid,
                scoring=scoring,
                n_jobs=n_jobs,
                random_state=random_state,
                option="bagging",
            )

            finish(
                runner,
                X_train,
                y_train,
                X_test,
                y_test,
                smote_class,
            )

Performing BorderlineSMOTE with DecisionTreeClassifier on ecoli take 1


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Performing BorderlineSMOTE with DecisionTreeClassifier on ecoli take 2


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Process SpawnPoolWorker-15:
Process SpawnPoolWorker-12:
Process SpawnPoolWorker-9:
Process SpawnPoolWorker-16:
Process SpawnPoolWorker-14:
Process SpawnPoolWorker-10:


KeyboardInterrupt: 

# Smote + Bagging different for each model

In [14]:
model_name = "DecisionTreeClassifier"

for smote_name, smote_class in sorted(SMOTE_CLASSES.items(), key=lambda x: x[0]):
    for dataset_name, (
        (X_train, y_train),
        (_, _),
        (X_test, y_test),
    ) in data_manager.serve():
        for take in range(1, n_takes + 1):
            random_state = RANDOM_SEED + take
            filename = f"{dataset_name}__{smote_name}__{model_name}__bagging_advanced__{take}.json"
            if filename in performed_runs:
                print(
                    f"Already performed {smote_name} with {model_name} on {dataset_name} take {take}"
                )
                continue
            print(
                f"Performing {smote_name} with {model_name} on {dataset_name} take {take}"
            )

            bagging_classifier_class = partial(
                base_bagging_classifier_class,
                oversampler_class=smote_class,
            )

            runner = ParamRunner(
                base_estimator_class=tree_class,
                bagging_classifier_class=bagging_classifier_class,
                param_grid=bagging_param_grid,
                scoring=scoring,
                option="bagging",
                n_jobs=n_jobs,
                random_state=random_state,
            )

            finish(
                runner,
                X_train,
                y_train,
                X_test,
                y_test,
                smote_class,
            )

Performing BorderlineSMOTE with DecisionTreeClassifier on ecoli take 1


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Performing BorderlineSMOTE with DecisionTreeClassifier on ecoli take 2


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Performing BorderlineSMOTE with DecisionTreeClassifier on ecoli take 3


...

Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Performing BorderlineSMOTE with DecisionTreeClassifier on optical_digits take 1


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Process SpawnPoolWorker-111:
Process SpawnPoolWorker-104:
Process SpawnPoolWorker-107:
Process SpawnPoolWorker-109:
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/queues.py", line 365, in get
    res = self._reader.recv_bytes()
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/envs/automl/lib/python3.11/multiprocessing/connection.py", line 216, in recv_bytes
    buf = self._recv_bytes(maxlength)
          ^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

## Standard models

In [ ]:
for smote_name, smote_class in sorted(SMOTE_CLASSES.items(), key=lambda x: x[0]):
    for dataset_name, (
        (X_train, y_train),
        (_, _),
        (X_test, y_test),
    ) in data_manager.serve(oversampler=smote_class()):
        for model_name, model_class in (
            ("RandomForestClassifier", random_forest_class),
            ("XGBClassifier", xgb_class),
        ):
            for take in range(1, n_takes + 1):
                random_state = RANDOM_SEED + take
                filename = f"{dataset_name}__{smote_name}__{model_name}__no_bagging__{take}.json"
                if filename in performed_runs:
                    print(
                        f"Already performed {smote_name} with {model_name} on {dataset_name} take {take}"
                    )
                    continue
                print(
                    f"Performing {smote_name} with {model_name} on {dataset_name} take {take}"
                )

                runner = ParamRunner(
                    base_estimator_class=model_class,
                    param_grid=bagging_param_grid,
                    scoring=scoring,
                    n_jobs=n_jobs,
                    random_state=random_state,
                    option="no_bagging",
                )

                finish(
                    runner,
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    smote_class,
                )

# output removed for clearity

Already performed BorderlineSMOTE with RandomForestClassifier on ecoli take 1
Already performed BorderlineSMOTE with RandomForestClassifier on ecoli take 2
Already performed BorderlineSMOTE with RandomForestClassifier on ecoli take 3
Already performed BorderlineSMOTE with RandomForestClassifier on ecoli take 4
Performing BorderlineSMOTE with RandomForestClassifier on ecoli take 5


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Performing BorderlineSMOTE with XGBClassifier on ecoli take 1


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Performing BorderlineSMOTE with XGBClassifier on ecoli take 2


Parameter Grid:   0%|          | 0/15 [00:00<?, ?it/s]

Process SpawnPoolWorker-19:
Process SpawnPoolWorker-21:
Process SpawnPoolWorker-22:
Process SpawnPoolWorker-17:
Process SpawnPoolWorker-24:


KeyboardInterrupt: 